# Solar Filament Segmentation: final reproducible pipeline

This notebook covers both selected final systems. A default **Run All** rebuilds the two submission CSV files from archived raw predictions and verifies their hashes. Set the training or inference switches below only when the official data and a CUDA GPU are available.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

def find_package_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'FINAL_SUBMISSION_MANIFEST.json').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside final_submission_package')

root_override = os.environ.get('SOLAR_PACKAGE_ROOT')
ROOT = Path(root_override).resolve() if root_override else find_package_root(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get('SOLAR_DATA_ROOT', ROOT / 'data/MAGFiLO_1.0_Kaggle_2026')).resolve()
WORK_DIR = Path(os.environ.get('SOLAR_WORK_DIR', ROOT / 'work')).resolve()
DEVICE = os.environ.get('SOLAR_DEVICE', '0')
VARIANTS = ('y17_v', 'qt01_ttah')
RUN_PREPARE = False
RUN_TRAINING = False
RUN_GPU_INFERENCE = False
REBUILD_CSV = True
print('package root:', ROOT)
print('data root:', DATA_ROOT)

In [ ]:
import torch
import torchvision
import ultralytics
import yaml
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('ultralytics:', ultralytics.__version__)
print('cuda available:', torch.cuda.is_available())

## 1. Data preparation
Grouped folds prevent repeated annotation passes of the same physical observation from leaking across folds. Majority-consensus aggregation then produces one training image per physical observation.

In [ ]:
def run_cli(*arguments: str) -> None:
    command = [sys.executable, str(ROOT / 'scripts/run_final_solution.py'), *arguments]
    print('RUN', ' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)

if RUN_PREPARE or RUN_TRAINING or RUN_GPU_INFERENCE or REBUILD_CSV:
    annotation = DATA_ROOT / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'
    test_images = DATA_ROOT / 'test/test_images'
    if not annotation.is_file() or not test_images.is_dir():
        raise FileNotFoundError('Official competition data not found under ' + str(DATA_ROOT))
if RUN_PREPARE:
    run_cli('--variant', 'y17_v', '--stage', 'prepare', '--data-root', str(DATA_ROOT), '--work-dir', str(WORK_DIR))

## 2. Optional full-data training
Y17-V uses the standard segmentation objective for 22 epochs. QT01 uses 26 epochs and mixes detached predicted mask IoU 50:50 into positive classification targets; box and mask losses remain unchanged. Exact settings are in `configs/final/`.

In [ ]:
if RUN_TRAINING:
    for variant in VARIANTS:
        run_cli('--variant', variant, '--stage', 'train', '--data-root', str(DATA_ROOT),
                '--work-dir', str(WORK_DIR), '--device', DEVICE)

## 3. Optional checkpoint inference and deterministic post-processing
Inference runs the variant-specific flip TTA, restores masks to original coordinates, removes duplicate masks at IoU 0.50, applies the frozen confidence and area thresholds, resolves overlaps in descending-score order, and writes COCO RLE.

In [ ]:
if RUN_GPU_INFERENCE:
    for variant in VARIANTS:
        run_cli('--variant', variant, '--stage', 'predict', '--data-root', str(DATA_ROOT),
                '--work-dir', str(WORK_DIR), '--device', DEVICE)

if REBUILD_CSV:
    for variant in VARIANTS:
        run_cli('--variant', variant, '--stage', 'rebuild-csv', '--data-root', str(DATA_ROOT),
                '--work-dir', str(WORK_DIR))

## 4. Integrity verification
The package verifier checks the submissions, checkpoints, raw predictions, audit results, Notebook structure, and SHA256 hashes. Publication-only blockers such as pending public URLs are reported separately by `--publication-ready`.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts/verify_package.py')], cwd=ROOT, check=True)
manifest = json.loads((ROOT / 'FINAL_SUBMISSION_MANIFEST.json').read_text(encoding='utf-8'))
for item in manifest['final_submissions']:
    print(item['variant'], 'OOF mean PQ =', item['twofold_mean_pq'],
          'public PQ =', item['public_pq'], 'role =', item['role'])